# CS3807 – Deep Learning Laboratory
# Experiment 6: End-to-End Study of RNN, LSTM and GRU for Sequence Learning and Video Understanding

**B.Tech AI & DS, Semester V — AY 2026–27**

This notebook implements the complete experiment:
1. UCI HAR sequence classification with Vanilla RNN / LSTM / GRU
2. Backpropagation Through Time — numerical exercise
3. Training curves, evaluation metrics, confusion matrices
4. Effect of sequence length
5. Video understanding: CNN (MobileNetV2) + LSTM/GRU on UCF101 subset
6. Sequence-to-sequence learning (reversal task) with encoder–decoder LSTM
7. Consolidated results
8. Additional exercises (16/64 units, GRU vs LSTM, stacked RNN, BiLSTM, seq-length cost, video LSTM vs GRU, variable-length seq2seq)

**All figures are saved as `.eps` files in the working (root) directory, typeset in Times New Roman using a font file named `times.ttf` that must be placed in the root directory before running this notebook.**


## 0. Setup — imports, reproducibility, EPS/Times New Roman plotting config

In [ ]:

import os, io, time, glob, zipfile, random, urllib.request
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safe for headless/EPS export; remove if you want inline display too
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- Times New Roman + EPS configuration -------------------------------
# times.ttf MUST be present in the root (working) directory.
FONT_PATH = "times.ttf"
if os.path.exists(FONT_PATH):
    fm.fontManager.addfont(FONT_PATH)
    TIMES_NAME = fm.FontProperties(fname=FONT_PATH).get_name()
else:
    print("WARNING: times.ttf not found in root directory — falling back to 'serif'. "
          "Place times.ttf next to this notebook for exact Times New Roman rendering.")
    TIMES_NAME = "serif"

plt.rcParams["font.family"] = TIMES_NAME
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = TIMES_NAME
plt.rcParams["mathtext.it"] = TIMES_NAME
plt.rcParams["mathtext.bf"] = TIMES_NAME
plt.rcParams["savefig.format"] = "eps"
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

ROOT_DIR = "."  # figures are saved here, i.e. the root/working directory

def save_fig(fig, name):
    """Save a matplotlib figure as EPS (Times New Roman) in the root directory."""
    path = os.path.join(ROOT_DIR, f"{name}.eps")
    fig.savefig(path, format="eps", bbox_inches="tight")
    print(f"Saved: {path}")

print("TensorFlow:", tf.__version__)
print("Using font:", TIMES_NAME)


TensorFlow: 2.20.0
Using font: Times New Roman


## 1. Dataset — UCI Human Activity Recognition Using Smartphones (raw inertial signals)

Downloads and extracts the dataset if not already present, then loads the **raw 128×9 inertial
signal windows** (not the 561-feature vectors), per the experiment spec.

In [ ]:

UCI_HAR_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
DATA_DIR = "UCI_HAR_Dataset"

def download_and_extract_har(dest=DATA_DIR):
    if os.path.isdir(dest):
        print("Dataset already extracted at", dest)
        return dest
    zpath = "har.zip"
    if not os.path.exists(zpath):
        print("Downloading UCI HAR dataset ...")
        urllib.request.urlretrieve(UCI_HAR_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
    # The archive nests a further zip / folder named "UCI HAR Dataset"
    inner_zip = glob.glob(os.path.join(dest, "**", "*.zip"), recursive=True)
    for iz in inner_zip:
        with zipfile.ZipFile(iz) as z:
            z.extractall(os.path.dirname(iz))
    return dest

def find_har_root(dest=DATA_DIR):
    for root, dirs, files in os.walk(dest):
        if "Inertial Signals" in dirs or any(d == "Inertial Signals" for d in dirs):
            pass
    for root, dirs, files in os.walk(dest):
        if os.path.basename(root) == "train" and "Inertial Signals" in dirs:
            return os.path.dirname(root)
    raise FileNotFoundError("Could not locate 'UCI HAR Dataset' root after extraction.")

download_and_extract_har()
HAR_ROOT = find_har_root()
print("HAR root:", HAR_ROOT)

SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

def load_signals(split):
    split_dir = os.path.join(HAR_ROOT, split, "Inertial Signals")
    channels = []
    for sig in SIGNALS:
        fpath = os.path.join(split_dir, f"{sig}_{split}.txt")
        arr = np.loadtxt(fpath)          # (N, 128)
        channels.append(arr)
    X = np.stack(channels, axis=-1)      # (N, 128, 9)
    return X

def load_labels(split):
    y = np.loadtxt(os.path.join(HAR_ROOT, split, f"y_{split}.txt")).astype(int) - 1  # 0-indexed, 6 classes
    return y

X_train_full = load_signals("train")
y_train_full = load_labels("train")
X_test_full  = load_signals("test")
y_test_full  = load_labels("test")

ACTIVITIES = ["WALKING", "WALKING_UPSTAIRS", "WALKING_DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]

print("Full train:", X_train_full.shape, "Full test:", X_test_full.shape)


HAR root: UCI_HAR_Dataset/UCI HAR Dataset
Full train: (7352, 128, 9) Full test: (2947, 128, 9)


In [ ]:

# ---- Recommended laboratory subset: 1500-3000 windows, all 6 classes, ~balanced ----
SUBSET_SIZE = 2400

def balanced_subset(X, y, n_total, seed=SEED):
    rng = np.random.RandomState(seed)
    n_classes = len(np.unique(y))
    per_class = n_total // n_classes
    idxs = []
    for c in range(n_classes):
        c_idx = np.where(y == c)[0]
        rng.shuffle(c_idx)
        idxs.extend(c_idx[:per_class])
    idxs = np.array(idxs)
    rng.shuffle(idxs)
    return X[idxs], y[idxs]

X_sub, y_sub = balanced_subset(X_train_full, y_train_full, SUBSET_SIZE)
print("Subset shape:", X_sub.shape, "class counts:", np.bincount(y_sub))


Subset shape: (2400, 128, 9) class counts: [400 400 400 400 400 400]


## 2. Preprocessing — split, normalize, verify

In [ ]:

# 70/15/15 split. First carve out test from the subset, then val from remaining train.
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sub, y_sub, test_size=0.30, stratify=y_sub, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

# Normalize using TRAINING statistics only, per channel.
mean = X_train.mean(axis=(0, 1), keepdims=True)
std  = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

def normalize(X):
    return (X - mean) / std

X_train_n, X_val_n, X_test_n = normalize(X_train), normalize(X_val), normalize(X_test)

N_train, T_STEPS, F_FEATS = X_train_n.shape
N_val, N_test = X_val_n.shape[0], X_test_n.shape[0]
N_CLASSES = len(ACTIVITIES)

print("Input tensor shape:")
print(f"  Training   : {X_train_n.shape}")
print(f"  Validation : {X_val_n.shape}")
print(f"  Testing    : {X_test_n.shape}")
print(f"Number of classes: {N_CLASSES}")
print(f"Number of features per time step: {F_FEATS}")
print(f"Sequence length: {T_STEPS}")

print("\nClass distribution (train):", np.bincount(y_train))
print("Class distribution (val):  ", np.bincount(y_val))
print("Class distribution (test): ", np.bincount(y_test))


Input tensor shape:
  Training   : (1680, 128, 9)
  Validation : (360, 128, 9)
  Testing    : (360, 128, 9)
Number of classes: 6
Number of features per time step: 9
Sequence length: 128

Class distribution (train): [280 280 280 280 280 280]
Class distribution (val):   [60 60 60 60 60 60]
Class distribution (test):  [60 60 60 60 60 60]


## 3. Temporal Data Visualization — Plot 1

In [ ]:

def plot_activity_signals(X, y, activities, channel_names, n_activities=3, channels_to_plot=(0, 3, 6)):
    fig, axes = plt.subplots(n_activities, 1, figsize=(9, 3 * n_activities), sharex=True)
    chosen_classes = np.linspace(0, len(activities) - 1, n_activities).astype(int)
    for ax, c in zip(axes, chosen_classes):
        idx = np.where(y == c)[0][0]
        for ch in channels_to_plot:
            ax.plot(range(1, X.shape[1] + 1), X[idx, :, ch], label=channel_names[ch], linewidth=1.2)
        ax.set_title(f"Activity: {activities[c]}")
        ax.set_ylabel("Sensor value")
        ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("Time step (1-128)")
    fig.suptitle("Plot 1: Sensor Signal vs Time (representative channels)", y=1.02)
    fig.tight_layout()
    return fig

fig1 = plot_activity_signals(X_train, y_train, ACTIVITIES, SIGNALS, n_activities=3)
save_fig(fig1, "plot1_sensor_signals_vs_time")
plt.close(fig1)


Saved: ./plot1_sensor_signals_vs_time.eps


**Required inference (Plot 1):**
1. *Temporal pattern:* Dynamic activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS) show strong periodic oscillations in the acceleration/gyroscope channels, while static activities (SITTING, STANDING, LAYING) show near-flat signals with only small noise fluctuations.
2. *Similar patterns:* The three walking-type activities look visually similar to each other (periodic, high-amplitude); the three static activities also resemble one another (low-amplitude, near-constant).
3. *Why order matters:* The class identity is encoded in the shape/periodicity of the signal across time — shuffling the 128 time steps would destroy the oscillation frequency and phase information a recurrent model needs, even though the set of individual values would be unchanged.

## 4. Vanilla RNN — Architecture

In [ ]:

def build_rnn(units=32, input_shape=(T_STEPS, F_FEATS), n_classes=N_CLASSES, dropout=0.2):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.SimpleRNN(units),
        layers.Dropout(dropout),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ], name="Vanilla_RNN")
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

build_rnn().summary()


Model: "Vanilla_RNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           102 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,974 (7.71 KB)

 Trainable params: 1,974 (7.71 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Backpropagation Through Time — Numerical Exercise

In [ ]:

x = [0.5, 0.7, 0.2]
h = 0.0
Wx, Wh, b = 0.5, 0.8, 0.1

hs = []
for t, xt in enumerate(x, start=1):
    h = np.tanh(Wx * xt + Wh * h + b)
    hs.append(h)
    print(f"h{t} = tanh({Wx}*{xt} + {Wh}*{h if t==1 else hs[t-2]:.6f} + {b}) = {h:.6f}")

print("\nManual results: h1={:.6f}, h2={:.6f}, h3={:.6f}".format(*hs))


h1 = tanh(0.5*0.5 + 0.8*0.336376 + 0.1) = 0.336376
h2 = tanh(0.5*0.7 + 0.8*0.336376 + 0.1) = 0.616352
h3 = tanh(0.5*0.2 + 0.8*0.616352 + 0.1) = 0.599958

Manual results: h1=0.336376, h2=0.616352, h3=0.599958


**Required inference:** Compare `h1, h2, h3` printed above with the values you compute by
hand using $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$. They should match to numerical precision.

## 6. Training Utility — shared across RNN/LSTM/GRU (and later variants)

In [ ]:

def train_and_time(model, X_tr, y_tr, X_va, y_va, epochs=30, batch_size=32, verbose=0):
    es = callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                      epochs=epochs, batch_size=batch_size, callbacks=[es], verbose=verbose)
    train_time = time.time() - t0
    return hist, train_time

def plot_loss_acc(hist, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(hist.history["loss"], label="Train loss")
    axes[0].plot(hist.history["val_loss"], label="Val loss")
    axes[0].set_title(f"{model_name}: Loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(np.array(hist.history["accuracy"]) * 100, label="Train acc")
    axes[1].plot(np.array(hist.history["val_accuracy"]) * 100, label="Val acc")
    axes[1].set_title(f"{model_name}: Accuracy")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].legend()
    fig.tight_layout()
    return fig

def evaluate_model(model, X_te, y_te, class_names, model_name):
    y_prob = model.predict(X_te, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_te, y_pred) * 100
    prec = precision_score(y_te, y_pred, average="macro", zero_division=0) * 100
    rec  = recall_score(y_te, y_pred, average="macro", zero_division=0) * 100
    f1   = f1_score(y_te, y_pred, average="macro", zero_division=0) * 100
    cm   = confusion_matrix(y_te, y_pred)
    n_params = model.count_params()
    return {"model": model_name, "accuracy": acc, "precision": prec, "recall": rec,
            "f1": f1, "confusion_matrix": cm, "params": n_params, "y_pred": y_pred}

def plot_confusion(cm, class_names, model_name):
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(f"Plot 4: Confusion Matrix — {model_name}")
    fig.tight_layout()
    return fig


## 7. Train RNN, LSTM, GRU (identical settings) — Plots 2 & 3, evaluation, Plot 4

In [ ]:

results = {}
histories = {}
train_times = {}
models_dict = {}

builders = {
    "RNN":  lambda: build_rnn(),
    "LSTM": lambda: models.Sequential([
                layers.Input(shape=(T_STEPS, F_FEATS)),
                layers.LSTM(32), layers.Dropout(0.2),
                layers.Dense(16, activation="relu"),
                layers.Dense(N_CLASSES, activation="softmax")], name="LSTM"),
    "GRU":  lambda: models.Sequential([
                layers.Input(shape=(T_STEPS, F_FEATS)),
                layers.GRU(32), layers.Dropout(0.2),
                layers.Dense(16, activation="relu"),
                layers.Dense(N_CLASSES, activation="softmax")], name="GRU"),
}

for name, build_fn in builders.items():
    print(f"\n=== Training {name} ===")
    m = build_fn()
    m.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    hist, ttime = train_and_time(m, X_train_n, y_train, X_val_n, y_val, epochs=30, batch_size=32)
    fig = plot_loss_acc(hist, name)
    save_fig(fig, f"plot2_3_{name.lower()}_loss_accuracy")
    plt.close(fig)

    res = evaluate_model(m, X_test_n, y_test, ACTIVITIES, name)
    res["training_time_s"] = ttime
    fig_cm = plot_confusion(res["confusion_matrix"], ACTIVITIES, name)
    save_fig(fig_cm, f"plot4_confusion_matrix_{name.lower()}")
    plt.close(fig_cm)

    print(f"{name}: Acc={res['accuracy']:.2f}% F1={res['f1']:.2f}% "
          f"Params={res['params']} Time={ttime:.1f}s")

    results[name] = res
    histories[name] = hist
    train_times[name] = ttime
    models_dict[name] = m



=== Training RNN ===


Saved: ./plot2_3_rnn_loss_accuracy.eps
Saved: ./plot4_confusion_matrix_rnn.eps
RNN: Acc=77.78% F1=77.78% Params=1974 Time=26.7s

=== Training LSTM ===


Saved: ./plot2_3_lstm_loss_accuracy.eps
Saved: ./plot4_confusion_matrix_lstm.eps
LSTM: Acc=96.11% F1=96.11% Params=6006 Time=23.5s

=== Training GRU ===


Saved: ./plot2_3_gru_loss_accuracy.eps
Saved: ./plot4_confusion_matrix_gru.eps
GRU: Acc=95.28% F1=95.28% Params=4758 Time=19.4s


**Convergence inferences:**
- *RNN:* Tends to converge more slowly and can plateau at a lower validation accuracy; the gap between train and val curves widens if the model starts memorizing short-term noise instead of long-range periodicity.
- *LSTM:* Generally converges smoothly with a shrinking train/val loss gap, since the cell state preserves longer-range dependencies (e.g., the full gait cycle in WALKING).
- *GRU:* Similar or slightly faster convergence than LSTM with fewer parameters, since it merges the input/forget gates into a single update gate.

**Confusion-matrix inference:** WALKING_UPSTAIRS and WALKING_DOWNSTAIRS are typically the most confused pair (similar periodic acceleration signatures), while LAYING is usually the most cleanly separated (near-zero dynamic acceleration). Static-activity confusions (SITTING vs STANDING) can also appear because both have very low-magnitude, similar-looking signals.

## 8. RNN vs LSTM vs GRU — Comparison Table & Plot 5

In [ ]:

comparison_rows = []
for name in ["RNN", "LSTM", "GRU"]:
    r = results[name]
    comparison_rows.append({
        "Model": name,
        "Accuracy (%)": round(r["accuracy"], 2),
        "Macro Precision (%)": round(r["precision"], 2),
        "Macro Recall (%)": round(r["recall"], 2),
        "Macro F1 (%)": round(r["f1"], 2),
        "Parameters": r["params"],
        "Training Time (s)": round(r["training_time_s"], 1),
    })
comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

# Property table (gates etc.) merged with measured values
property_df = pd.DataFrame({
    "Property": ["Hidden state", "Cell state", "Forget gate", "Input gate",
                 "Output gate", "Update gate", "Reset gate",
                 "Parameters", "Training time (s)", "Test F1-score (%)"],
    "RNN":  ["Yes", "No", "No", "No", "No", "No", "No",
             results["RNN"]["params"], round(results["RNN"]["training_time_s"], 1), round(results["RNN"]["f1"], 2)],
    "LSTM": ["Yes", "Yes", "Yes", "Yes", "Yes", "No", "No",
             results["LSTM"]["params"], round(results["LSTM"]["training_time_s"], 1), round(results["LSTM"]["f1"], 2)],
    "GRU":  ["Yes", "No", "No", "No", "No", "Yes", "Yes",
             results["GRU"]["params"], round(results["GRU"]["training_time_s"], 1), round(results["GRU"]["f1"], 2)],
})
print("\n", property_df.to_string(index=False))


Model  Accuracy (%)  Macro Precision (%)  Macro Recall (%)  Macro F1 (%)  Parameters  Training Time (s)
  RNN         77.78                77.92             77.78         77.78        1974               26.7
 LSTM         96.11                96.15             96.11         96.11        6006               23.5
  GRU         95.28                95.31             95.28         95.28        4758               19.4

          Property    RNN   LSTM    GRU
     Hidden state    Yes    Yes    Yes
       Cell state     No    Yes     No
      Forget gate     No    Yes     No
       Input gate     No    Yes     No
      Output gate     No    Yes     No
      Update gate     No     No    Yes
       Reset gate     No     No    Yes
       Parameters   1974   6006   4758
Training time (s)   26.7   23.5   19.4
Test F1-score (%)  77.78  96.11  95.28


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
metrics_to_plot = ["Accuracy (%)", "Macro F1 (%)"]
x = np.arange(len(comparison_df))
width = 0.35
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, comparison_df[metric], width, label=metric)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(comparison_df["Model"])
ax.set_ylabel("Percent")
ax.set_title("Plot 5: Model Performance Comparison")
ax.legend()
fig.tight_layout()
save_fig(fig, "plot5_model_performance_comparison")
plt.close(fig)


Saved: ./plot5_model_performance_comparison.eps


**Trade-off inference:** LSTM typically has the most parameters and longest training time
because of its four gates/cell state, RNN the fewest parameters but often the weakest accuracy,
and GRU sits in between — usually reaching accuracy close to LSTM with a noticeably smaller
parameter count and faster training, illustrating the Predictive Performance ↔ Complexity ↔
Training Cost trade-off.

## 9. Effect of Sequence Length — Plot 6

In [ ]:

def truncate_seq(X, T):
    return X[:, :T, :]

seq_lengths = [32, 64, 128]
seqlen_results = {m: {} for m in ["RNN", "LSTM", "GRU"]}

for T in seq_lengths:
    Xtr_T = truncate_seq(X_train_n, T)
    Xva_T = truncate_seq(X_val_n, T)
    Xte_T = truncate_seq(X_test_n, T)

    variant_builders = {
        "RNN":  lambda: models.Sequential([layers.Input((T, F_FEATS)), layers.SimpleRNN(32),
                                            layers.Dropout(0.2), layers.Dense(16, activation="relu"),
                                            layers.Dense(N_CLASSES, activation="softmax")]),
        "LSTM": lambda: models.Sequential([layers.Input((T, F_FEATS)), layers.LSTM(32),
                                            layers.Dropout(0.2), layers.Dense(16, activation="relu"),
                                            layers.Dense(N_CLASSES, activation="softmax")]),
        "GRU":  lambda: models.Sequential([layers.Input((T, F_FEATS)), layers.GRU(32),
                                            layers.Dropout(0.2), layers.Dense(16, activation="relu"),
                                            layers.Dense(N_CLASSES, activation="softmax")]),
    }
    for name, bfn in variant_builders.items():
        m = bfn()
        m.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        hist, ttime = train_and_time(m, Xtr_T, y_train, Xva_T, y_val, epochs=20)
        r = evaluate_model(m, Xte_T, y_test, ACTIVITIES, f"{name}_T{T}")
        seqlen_results[name][T] = {"f1": r["f1"], "time_s": ttime, "params": m.count_params()}
        print(f"T={T:3d} {name}: F1={r['f1']:.2f}% time={ttime:.1f}s")

seqlen_df = pd.DataFrame({
    "Sequence Length": seq_lengths,
    "RNN F1":  [round(seqlen_results["RNN"][T]["f1"], 2) for T in seq_lengths],
    "LSTM F1": [round(seqlen_results["LSTM"][T]["f1"], 2) for T in seq_lengths],
    "GRU F1":  [round(seqlen_results["GRU"][T]["f1"], 2) for T in seq_lengths],
})
print("\n", seqlen_df.to_string(index=False))


T= 32 RNN: F1=77.50% time=13.5s
T= 32 LSTM: F1=93.60% time=11.8s
T= 32 GRU: F1=93.10% time=11.8s
T= 64 RNN: F1=76.95% time=14.2s
T= 64 LSTM: F1=95.27% time=11.8s
T= 64 GRU: F1=94.71% time=12.6s
T=128 RNN: F1=80.63% time=17.8s
T=128 LSTM: F1=94.14% time=13.9s
T=128 GRU: F1=94.97% time=14.2s

  Sequence Length  RNN F1  LSTM F1  GRU F1
              32   77.50    93.60   93.10
              64   76.95    95.27   94.71
             128   80.63    94.14   94.97


In [ ]:

fig, ax = plt.subplots(figsize=(7, 5))
for name in ["RNN", "LSTM", "GRU"]:
    ax.plot(seq_lengths, [seqlen_results[name][T]["f1"] for T in seq_lengths], marker="o", label=name)
ax.set_xlabel("Sequence length (T)")
ax.set_ylabel("Test F1-score (%)")
ax.set_title("Plot 6: Sequence Length vs Test F1-score")
ax.legend()
fig.tight_layout()
save_fig(fig, "plot6_seqlen_vs_f1")
plt.close(fig)


Saved: ./plot6_seqlen_vs_f1.eps


**Inference:** As sequence length increases from 32 to 128, more temporal context (e.g. a
full gait cycle) becomes visible to the model, which typically improves F1-score, especially for
dynamic activities. Very short windows (T=32) may not contain enough of the periodic pattern to
disambiguate similar dynamic activities, so accuracy tends to be lower there.

## 10. Video Understanding — CNN (MobileNetV2) + LSTM/GRU

Uses a small subset of UCF101. **You must download UCF101 separately**
(https://www.crcv.ucf.edu/data/UCF101.php) and set `UCF101_DIR` below to a folder that contains
per-class subfolders of `.avi`/`.mp4` files for a handful of selected classes.

In [ ]:

import cv2  # opencv-python

UCF101_DIR = "UCF101_subset"     # expected structure: UCF101_subset/<ClassName>/*.avi
SELECTED_CLASSES = ["Basketball", "Biking", "Walking", "Running", "TennisSwing"]
N_FRAMES = 10
FRAME_SIZE = (224, 224)

def sample_frames(video_path, n_frames=N_FRAMES, size=FRAME_SIZE):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.linspace(0, max(total - 1, 0), n_frames).astype(int)
    frames = []
    for i in range(total):
        ret, frame = cap.read()
        if not ret:
            break
        if i in idxs:
            frame = cv2.resize(frame, size)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
    cap.release()
    if len(frames) < n_frames:
        return None
    return np.stack(frames[:n_frames])  # (10, 224, 224, 3)

def build_video_dataset(root_dir, classes, n_frames=N_FRAMES):
    X, y, labels = [], [], []
    for ci, cls in enumerate(classes):
        cls_dir = os.path.join(root_dir, cls)
        if not os.path.isdir(cls_dir):
            print(f"Skipping missing class folder: {cls_dir}")
            continue
        for vid in glob.glob(os.path.join(cls_dir, "*")):
            frames = sample_frames(vid, n_frames)
            if frames is not None:
                X.append(frames)
                y.append(ci)
        labels.append(cls)
    if not X:
        raise FileNotFoundError(
            f"No videos found under {root_dir}. Populate UCF101_DIR with per-class folders first.")
    return np.array(X), np.array(y), labels

VIDEO_DATA_AVAILABLE = os.path.isdir(UCF101_DIR)
if VIDEO_DATA_AVAILABLE:
    Xv_raw, yv, video_classes = build_video_dataset(UCF101_DIR, SELECTED_CLASSES)
    print("Video frames tensor:", Xv_raw.shape)
else:
    print(f"'{UCF101_DIR}' not found — download a UCF101 subset and re-run this cell "
          f"before proceeding with the video-understanding section.")


'UCF101_subset' not found — download a UCF101 subset and re-run this cell before proceeding with the video-understanding section.


In [ ]:

# ---- CNN feature extraction (MobileNetV2, frozen, no head) ----
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def build_cnn_extractor():
    base = MobileNetV2(include_top=False, weights="imagenet", pooling="avg",
                        input_shape=(224, 224, 3))
    base.trainable = False  # freeze — do not train the CNN from scratch
    return base

if VIDEO_DATA_AVAILABLE:
    cnn_extractor = build_cnn_extractor()
    D = cnn_extractor.output_shape[-1]
    print(f"CNN feature dimension D = {D}")

    def extract_features(X_frames, extractor, batch_size=16):
        n, t, h, w, c = X_frames.shape
        flat = X_frames.reshape(n * t, h, w, c).astype("float32")
        flat = preprocess_input(flat)
        feats = extractor.predict(flat, batch_size=batch_size, verbose=0)
        return feats.reshape(n, t, -1)  # (N, 10, D)

    Xv_feats = extract_features(Xv_raw, cnn_extractor)
    print("Feature sequence tensor supplied to recurrent network:", Xv_feats.shape)  # (N, 10, D)

    Xv_tr, Xv_te, yv_tr, yv_te = train_test_split(Xv_feats, yv, test_size=0.2,
                                                   stratify=yv, random_state=SEED)
    Xv_tr, Xv_va, yv_tr, yv_va = train_test_split(Xv_tr, yv_tr, test_size=0.2,
                                                   stratify=yv_tr, random_state=SEED)


In [ ]:

if VIDEO_DATA_AVAILABLE:
    # Plot 7: representative sampled frames from one video
    fig, axes = plt.subplots(1, N_FRAMES, figsize=(2 * N_FRAMES, 2.2))
    for i, ax in enumerate(axes):
        ax.imshow(Xv_raw[0, i])
        ax.axis("off")
        ax.set_title(f"t={i+1}", fontsize=8)
    fig.suptitle(f"Plot 7: Sampled Frames — class '{video_classes[yv[0]]}'")
    fig.tight_layout()
    save_fig(fig, "plot7_video_sample_frames")
    plt.close(fig)


In [ ]:

def build_video_rnn(recurrent="LSTM", units=32, D=None, n_classes=None):
    Layer = layers.LSTM if recurrent == "LSTM" else layers.GRU
    model = models.Sequential([
        layers.Input(shape=(N_FRAMES, D)),
        Layer(units),
        layers.Dense(n_classes, activation="softmax"),
    ], name=f"CNN_{recurrent}")
    model.compile(optimizer=optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

video_results = {}
if VIDEO_DATA_AVAILABLE:
    n_cls_v = len(video_classes)
    for rec in ["LSTM", "GRU"]:
        vm = build_video_rnn(rec, D=D, n_classes=n_cls_v)
        vhist, vtime = train_and_time(vm, Xv_tr, yv_tr, Xv_va, yv_va, epochs=25, batch_size=8)
        vfig = plot_loss_acc(vhist, f"CNN-{rec} (video)")
        save_fig(vfig, f"plot8_video_{rec.lower()}_loss_accuracy")
        plt.close(vfig)

        vres = evaluate_model(vm, Xv_te, yv_te, video_classes, f"CNN-{rec}")
        vres["training_time_s"] = vtime
        vfig_cm = plot_confusion(vres["confusion_matrix"], video_classes, f"CNN-{rec}")
        save_fig(vfig_cm, f"plot9_video_confusion_{rec.lower()}")
        plt.close(vfig_cm)

        video_results[rec] = vres
        acc_v = vres["accuracy"]; f1_v = vres["f1"]
        print(f"CNN-{rec}: Acc={acc_v:.2f}% F1={f1_v:.2f}%")

        # Example predictions in the required format
        probs = vm.predict(Xv_te[:3], verbose=0)
        for i in range(3):
            pred_c = np.argmax(probs[i])
            print(f"Predicted class : {video_classes[pred_c]} | "
                  f"Actual class : {video_classes[yv_te[i]]} | "
                  f"Confidence : {probs[i][pred_c]:.2f}")


**Inference:** The CNN (MobileNetV2, frozen) captures per-frame *spatial* appearance —
object/scene context such as a basketball court or a bicycle — while the recurrent layer models
the *temporal* evolution of those per-frame features across the 10 sampled frames, e.g. the
repetitive motion pattern of running or a tennis swing.

## 11. Sequence-to-Sequence Learning — Synthetic Reversal Task

In [ ]:

VOCAB_MIN, VOCAB_MAX = 0, 9         # integer range
SEQ_LEN = 4
N_SAMPLES = 20000
START_TOKEN = VOCAB_MAX + 1         # 10, used to seed the decoder
VOCAB_SIZE = VOCAB_MAX + 2          # 0..9 plus start token

def make_reversal_dataset(n_samples, seq_len, seed=SEED):
    rng = np.random.RandomState(seed)
    X = rng.randint(VOCAB_MIN, VOCAB_MAX + 1, size=(n_samples, seq_len))
    Y = X[:, ::-1]
    return X, Y

X_seq, Y_seq = make_reversal_dataset(N_SAMPLES, SEQ_LEN)

# Encoder input: X_seq. Decoder input: START token + Y[:-1] (teacher forcing). Decoder target: Y.
decoder_input = np.concatenate([np.full((N_SAMPLES, 1), START_TOKEN), Y_seq[:, :-1]], axis=1)

Xtr_e, Xte_e, Xtr_d, Xte_d, Ytr, Yte = train_test_split(
    X_seq, decoder_input, Y_seq, test_size=0.15, random_state=SEED)
Xtr_e, Xva_e, Xtr_d, Xva_d, Ytr, Yva = train_test_split(
    Xtr_e, Xtr_d, Ytr, test_size=0.15, random_state=SEED)

print("Encoder input:", Xtr_e.shape, "Decoder input:", Xtr_d.shape, "Target:", Ytr.shape)


Encoder input: (14450, 4) Decoder input: (14450, 4) Target: (14450, 4)


In [ ]:

EMBED_DIM = 16
LATENT_DIM = 64

# --- Encoder ---
enc_inputs = layers.Input(shape=(SEQ_LEN,), name="encoder_input")
enc_emb = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(enc_inputs)
enc_out, state_h, state_c = layers.LSTM(LATENT_DIM, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# --- Decoder (teacher forcing during training) ---
dec_inputs = layers.Input(shape=(SEQ_LEN,), name="decoder_input")
dec_emb_layer = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
dec_emb = dec_emb_layer(dec_inputs)
dec_lstm = layers.LSTM(LATENT_DIM, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(dec_emb, initial_state=encoder_states)
dec_dense = layers.Dense(VOCAB_SIZE, activation="softmax")
dec_pred = dec_dense(dec_out)

seq2seq_model = models.Model([enc_inputs, dec_inputs], dec_pred, name="Seq2Seq_Encoder_Decoder")
seq2seq_model.compile(optimizer=optimizers.Adam(1e-3),
                       loss="sparse_categorical_crossentropy", metrics=["accuracy"])
seq2seq_model.summary()


Model: "Seq2Seq_Encoder_Decoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 16)     │        176 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 4, 16)     │        176 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 64),      │     20,736 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 4, 64),   │     20,736 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm_4[0][1],     │
│                     │ (None, 64)]       │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 4, 11)     │        715 │ lstm_5[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 42,539 (166.17 KB)

 Trainable params: 42,539 (166.17 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:

s2s_hist, s2s_time = train_and_time(seq2seq_model, [Xtr_e, Xtr_d], Ytr,
                                     [Xva_e, Xva_d], Yva, epochs=20, batch_size=64)
fig = plot_loss_acc(s2s_hist, "Seq2Seq (reversal)")
save_fig(fig, "plot_seq2seq_loss_accuracy")
plt.close(fig)


Saved: ./plot_seq2seq_loss_accuracy.eps


In [ ]:

# ---- Inference-time encoder/decoder (no teacher forcing) ----
encoder_model = models.Model(enc_inputs, encoder_states)

dec_state_h_in = layers.Input(shape=(LATENT_DIM,))
dec_state_c_in = layers.Input(shape=(LATENT_DIM,))
dec_single_in = layers.Input(shape=(1,))
dec_single_emb = dec_emb_layer(dec_single_in)
dec_single_out, dh, dc = dec_lstm(dec_single_emb, initial_state=[dec_state_h_in, dec_state_c_in])
dec_single_pred = dec_dense(dec_single_out)
decoder_model = models.Model([dec_single_in, dec_state_h_in, dec_state_c_in],
                              [dec_single_pred, dh, dc])

def decode_sequence(input_seq, max_len=SEQ_LEN):
    states = encoder_model.predict(input_seq[None, :], verbose=0)
    target = np.array([[START_TOKEN]])
    output = []
    for _ in range(max_len):
        pred, h, c = decoder_model.predict([target, states[0], states[1]], verbose=0)
        tok = np.argmax(pred[0, -1, :])
        output.append(tok)
        target = np.array([[tok]])
        states = [h, c]
    return output

token_correct, token_total, seq_correct = 0, 0, 0
sample_rows = []
for i in range(len(Xte_e)):
    pred_seq = decode_sequence(Xte_e[i])
    true_seq = list(Yte[i])
    token_correct += sum(p == t for p, t in zip(pred_seq, true_seq))
    token_total += len(true_seq)
    seq_correct += int(pred_seq == true_seq)
    if i < 5:
        sample_rows.append({"Input": list(Xte_e[i]), "Predicted Output": pred_seq,
                             "Expected Output": true_seq})

token_accuracy = token_correct / token_total
sequence_accuracy = seq_correct / len(Xte_e)

print(pd.DataFrame(sample_rows).to_string(index=False))
print(f"\nToken accuracy    : {token_accuracy:.4f}")
print(f"Sequence accuracy : {sequence_accuracy:.4f}")
print(f"Final training loss: {s2s_hist.history['loss'][-1]:.4f}")
print(f"Final validation loss: {s2s_hist.history['val_loss'][-1]:.4f}")


       Input Predicted Output Expected Output
[0, 3, 9, 1]     [1, 9, 3, 0]    [1, 9, 3, 0]
[6, 8, 6, 9]     [9, 6, 8, 6]    [9, 6, 8, 6]
[2, 2, 6, 0]     [0, 6, 2, 2]    [0, 6, 2, 2]
[5, 7, 7, 6]     [6, 7, 7, 5]    [6, 7, 7, 5]
[2, 0, 7, 1]     [1, 7, 0, 2]    [1, 7, 0, 2]

Token accuracy    : 1.0000
Sequence accuracy : 1.0000
Final training loss: 0.0004
Final validation loss: 0.0004


**Inference:** Sequence accuracy can be substantially lower than token accuracy because a
sequence is only counted correct if *every* token in it is correct — a single wrong token
(often a later one, since decoder errors can compound) fails the whole sequence, whereas token
accuracy averages correctness independently per position.

## 12. Consolidated Results

In [ ]:

consolidated_rows = []
for name in ["RNN", "LSTM", "GRU"]:
    r = results[name]
    consolidated_rows.append({"Model": name, "Accuracy": round(r["accuracy"], 2),
                               "Precision": round(r["precision"], 2), "Recall": round(r["recall"], 2),
                               "F1": round(r["f1"], 2), "Parameters": r["params"]})
if video_results:
    for rec, r in video_results.items():
        consolidated_rows.append({"Model": f"CNN-{rec}", "Accuracy": round(r["accuracy"], 2),
                                   "Precision": round(r["precision"], 2), "Recall": round(r["recall"], 2),
                                   "F1": round(r["f1"], 2), "Parameters": r["params"]})
consolidated_df = pd.DataFrame(consolidated_rows)
print(consolidated_df.to_string(index=False))

seq2seq_table = pd.DataFrame([{
    "Task": "Seq2Seq (reversal)",
    "Token Accuracy": round(token_accuracy, 4),
    "Sequence Accuracy": round(sequence_accuracy, 4),
    "Train Loss": round(s2s_hist.history["loss"][-1], 4),
    "Val Loss": round(s2s_hist.history["val_loss"][-1], 4),
}])
print("\n", seq2seq_table.to_string(index=False))


Model  Accuracy  Precision  Recall    F1  Parameters
  RNN     77.78      77.92   77.78 77.78        1974
 LSTM     96.11      96.15   96.11 96.11        6006
  GRU     95.28      95.31   95.28 95.28        4758

               Task  Token Accuracy  Sequence Accuracy  Train Loss  Val Loss
Seq2Seq (reversal)             1.0                1.0      0.0004    0.0004


## 13. Additional Exercises

### 13.1 — Recurrent units: 16 vs 32 (baseline) vs 64

In [ ]:

unit_variants = [16, 32, 64]
unit_results = {"RNN": {}, "LSTM": {}, "GRU": {}}

def build_variant(cell, units):
    Layer = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[cell]
    m = models.Sequential([
        layers.Input((T_STEPS, F_FEATS)), Layer(units), layers.Dropout(0.2),
        layers.Dense(16, activation="relu"), layers.Dense(N_CLASSES, activation="softmax")])
    m.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for cell in ["RNN", "LSTM", "GRU"]:
    for u in unit_variants:
        m = build_variant(cell, u)
        _, ttime = train_and_time(m, X_train_n, y_train, X_val_n, y_val, epochs=20)
        r = evaluate_model(m, X_test_n, y_test, ACTIVITIES, f"{cell}_{u}u")
        unit_results[cell][u] = {"acc": r["accuracy"], "f1": r["f1"],
                                  "params": m.count_params(), "time_s": ttime}
        print(f"{cell} units={u:2d}: Acc={r['accuracy']:.2f}% F1={r['f1']:.2f}% "
              f"Params={m.count_params()} Time={ttime:.1f}s")

units_df = pd.DataFrame([
    {"Model": cell, "Units": u, **unit_results[cell][u]}
    for cell in unit_results for u in unit_variants
])
print("\n", units_df.to_string(index=False))


RNN units=16: Acc=70.56% F1=70.50% Params=790 Time=17.6s
RNN units=32: Acc=74.17% F1=74.15% Params=1974 Time=18.1s
RNN units=64: Acc=83.89% F1=83.84% Params=5878 Time=18.6s
LSTM units=16: Acc=92.22% F1=92.23% Params=2038 Time=13.9s
LSTM units=32: Acc=94.72% F1=94.68% Params=6006 Time=14.4s
LSTM units=64: Acc=94.44% F1=94.40% Params=20086 Time=14.0s
GRU units=16: Acc=92.50% F1=92.57% Params=1670 Time=14.0s
GRU units=32: Acc=94.17% F1=94.09% Params=4758 Time=13.7s
GRU units=64: Acc=94.72% F1=94.71% Params=15542 Time=13.8s

 Model  Units       acc        f1  params    time_s
  RNN     16 70.555556 70.497908     790 17.621282
  RNN     32 74.166667 74.154015    1974 18.079748
  RNN     64 83.888889 83.842273    5878 18.578063
 LSTM     16 92.222222 92.226689    2038 13.905880
 LSTM     32 94.722222 94.681746    6006 14.406346
 LSTM     64 94.444444 94.402506   20086 13.954249
  GRU     16 92.500000 92.571283    1670 14.021228
  GRU     32 94.166667 94.085138    4758 13.710943
  GRU     64 

### 13.2 — GRU vs LSTM at identical unit counts (direct head-to-head)

In [ ]:

gru_vs_lstm = pd.DataFrame([
    {"Units": u,
     "LSTM F1": round(unit_results["LSTM"][u]["f1"], 2),
     "GRU F1":  round(unit_results["GRU"][u]["f1"], 2),
     "LSTM Params": unit_results["LSTM"][u]["params"],
     "GRU Params":  unit_results["GRU"][u]["params"],
     "LSTM Time (s)": round(unit_results["LSTM"][u]["time_s"], 1),
     "GRU Time (s)":  round(unit_results["GRU"][u]["time_s"], 1)}
    for u in unit_variants
])
print(gru_vs_lstm.to_string(index=False))


 Units  LSTM F1  GRU F1  LSTM Params  GRU Params  LSTM Time (s)  GRU Time (s)
    16    92.23   92.57         2038        1670           13.9          14.0
    32    94.68   94.09         6006        4758           14.4          13.7
    64    94.40   94.71        20086       15542           14.0          13.8


### 13.3 — Stacked (2-layer) recurrent network

In [ ]:

def build_stacked(cell="LSTM", units=32):
    Layer = {"RNN": layers.SimpleRNN, "LSTM": layers.LSTM, "GRU": layers.GRU}[cell]
    m = models.Sequential([
        layers.Input((T_STEPS, F_FEATS)),
        Layer(units, return_sequences=True), layers.Dropout(0.2),
        Layer(units), layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax")], name=f"Stacked_{cell}")
    m.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

stacked_m = build_stacked("LSTM", 32)
_, stacked_time = train_and_time(stacked_m, X_train_n, y_train, X_val_n, y_val, epochs=25)
stacked_res = evaluate_model(stacked_m, X_test_n, y_test, ACTIVITIES, "Stacked_LSTM")
print(f"Stacked 2-layer LSTM: Acc={stacked_res['accuracy']:.2f}% F1={stacked_res['f1']:.2f}% "
      f"(single-layer LSTM F1={results['LSTM']['f1']:.2f}%)")


Stacked 2-layer LSTM: Acc=93.61% F1=93.52% (single-layer LSTM F1=96.11%)


### 13.4 — Bidirectional LSTM vs Unidirectional LSTM

In [ ]:

def build_bilstm(units=32):
    m = models.Sequential([
        layers.Input((T_STEPS, F_FEATS)),
        layers.Bidirectional(layers.LSTM(units)), layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(N_CLASSES, activation="softmax")], name="BiLSTM")
    m.compile(optimizer=optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

bilstm_m = build_bilstm(32)
_, bilstm_time = train_and_time(bilstm_m, X_train_n, y_train, X_val_n, y_val, epochs=25)
bilstm_res = evaluate_model(bilstm_m, X_test_n, y_test, ACTIVITIES, "BiLSTM")

print(f"Unidirectional LSTM: F1={results['LSTM']['f1']:.2f}% Params={results['LSTM']['params']} "
      f"Time={results['LSTM']['training_time_s']:.1f}s")
print(f"Bidirectional LSTM : F1={bilstm_res['f1']:.2f}% Params={bilstm_m.count_params()} "
      f"Time={bilstm_time:.1f}s")


Unidirectional LSTM: F1=96.11% Params=6006 Time=23.5s
Bidirectional LSTM : F1=95.25% Params=11894 Time=24.6s


### 13.5 — Sequence length vs computational cost (extends Section 9)

In [ ]:

cost_df = pd.DataFrame([
    {"Sequence Length": T,
     "RNN time (s)": round(seqlen_results["RNN"][T]["time_s"], 1),
     "LSTM time (s)": round(seqlen_results["LSTM"][T]["time_s"], 1),
     "GRU time (s)": round(seqlen_results["GRU"][T]["time_s"], 1),
     "RNN params": seqlen_results["RNN"][T]["params"],
     "LSTM params": seqlen_results["LSTM"][T]["params"],
     "GRU params": seqlen_results["GRU"][T]["params"]}
    for T in seq_lengths
])
print(cost_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
for name in ["RNN", "LSTM", "GRU"]:
    ax.plot(seq_lengths, [seqlen_results[name][T]["time_s"] for T in seq_lengths], marker="o", label=name)
ax.set_xlabel("Sequence length (T)")
ax.set_ylabel("Training time (s)")
ax.set_title("Additional Exercise: Sequence Length vs Training Time")
ax.legend()
fig.tight_layout()
save_fig(fig, "extra_seqlen_vs_training_time")
plt.close(fig)


 Sequence Length  RNN time (s)  LSTM time (s)  GRU time (s)  RNN params  LSTM params  GRU params
              32          13.5           11.8          11.8        1974         6006        4758
              64          14.2           11.8          12.6        1974         6006        4758
             128          17.8           13.9          14.2        1974         6006        4758
Saved: ./extra_seqlen_vs_training_time.eps


### 13.6 — Video task: LSTM vs GRU on identical CNN features (explicit comparison)

In [ ]:

if video_results:
    video_cmp = pd.DataFrame([
        {"Model": f"CNN-{rec}", "Accuracy (%)": round(r["accuracy"], 2),
         "F1 (%)": round(r["f1"], 2), "Params": r["params"],
         "Training Time (s)": round(r["training_time_s"], 1)}
        for rec, r in video_results.items()
    ])
    print(video_cmp.to_string(index=False))
else:
    print("Populate UCF101_DIR (Section 10) to run this comparison.")


Populate UCF101_DIR (Section 10) to run this comparison.


### 13.7 — Seq2Seq with a different output length than the input

In [ ]:

# Modified task: reverse only the first 3 of 4 input integers -> output length 3
OUT_LEN = 3

def make_variable_length_dataset(n_samples, in_len, out_len, seed=SEED):
    rng = np.random.RandomState(seed)
    X = rng.randint(VOCAB_MIN, VOCAB_MAX + 1, size=(n_samples, in_len))
    Y = X[:, :out_len][:, ::-1]   # reverse only the first out_len elements
    return X, Y

Xv2, Yv2 = make_variable_length_dataset(N_SAMPLES, SEQ_LEN, OUT_LEN)
dec_in_v2 = np.concatenate([np.full((N_SAMPLES, 1), START_TOKEN), Yv2[:, :-1]], axis=1)

Xtr2_e, Xte2_e, Xtr2_d, Xte2_d, Ytr2, Yte2 = train_test_split(
    Xv2, dec_in_v2, Yv2, test_size=0.15, random_state=SEED)
Xtr2_e, Xva2_e, Xtr2_d, Xva2_d, Ytr2, Yva2 = train_test_split(
    Xtr2_e, Xtr2_d, Ytr2, test_size=0.15, random_state=SEED)

enc_in2 = layers.Input(shape=(SEQ_LEN,))
enc_emb2 = layers.Embedding(VOCAB_SIZE, EMBED_DIM)(enc_in2)
_, sh2, sc2 = layers.LSTM(LATENT_DIM, return_state=True)(enc_emb2)

dec_in2 = layers.Input(shape=(OUT_LEN,))
dec_emb_layer2 = layers.Embedding(VOCAB_SIZE, EMBED_DIM)
dec_lstm2 = layers.LSTM(LATENT_DIM, return_sequences=True, return_state=True)
dec_out2, _, _ = dec_lstm2(dec_emb_layer2(dec_in2), initial_state=[sh2, sc2])
dec_pred2 = layers.Dense(VOCAB_SIZE, activation="softmax")(dec_out2)

model_v2 = models.Model([enc_in2, dec_in2], dec_pred2, name="Seq2Seq_variable_length")
model_v2.compile(optimizer=optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
_, v2_time = train_and_time(model_v2, [Xtr2_e, Xtr2_d], Ytr2, [Xva2_e, Xva2_d], Yva2,
                             epochs=20, batch_size=64)

pred = model_v2.predict([Xte2_e[:5], Xte2_d[:5]], verbose=0)
pred_tokens = np.argmax(pred, axis=-1)
for i in range(5):
    print(f"Input: {list(Xte2_e[i])} -> Predicted: {list(pred_tokens[i])} | Expected: {list(Yte2[i])}")


Input: [np.int64(0), np.int64(3), np.int64(9), np.int64(1)] -> Predicted: [np.int64(9), np.int64(3), np.int64(0)] | Expected: [np.int64(9), np.int64(3), np.int64(0)]
Input: [np.int64(6), np.int64(8), np.int64(6), np.int64(9)] -> Predicted: [np.int64(6), np.int64(8), np.int64(6)] | Expected: [np.int64(6), np.int64(8), np.int64(6)]
Input: [np.int64(2), np.int64(2), np.int64(6), np.int64(0)] -> Predicted: [np.int64(6), np.int64(2), np.int64(2)] | Expected: [np.int64(6), np.int64(2), np.int64(2)]
Input: [np.int64(5), np.int64(7), np.int64(7), np.int64(6)] -> Predicted: [np.int64(7), np.int64(7), np.int64(5)] | Expected: [np.int64(7), np.int64(7), np.int64(5)]
Input: [np.int64(2), np.int64(0), np.int64(7), np.int64(1)] -> Predicted: [np.int64(7), np.int64(0), np.int64(2)] | Expected: [np.int64(7), np.int64(0), np.int64(2)]


---
### End-of-experiment checklist
- All required tables (`comparison_df`, `property_df`, `seqlen_df`, `consolidated_df`,
  `seq2seq_table`, `units_df`, `gru_vs_lstm`, `cost_df`) are populated from **actual runs** —
  fill in your report from the printed DataFrames, not from assumed values.
- All figures (`plot1_*` … `plot9_*`, `extra_*`) are saved as `.eps` files in the root/working
  directory, typeset in Times New Roman via `times.ttf`.
- Write the required inferences (Section 26 of the manual) next to each corresponding plot in
  your final report.